In [1]:
import pandas as pd
import numpy as np
import getpass, joblib
from sqlalchemy import create_engine, URL
from sklearn.metrics import roc_auc_score

# frozen model
m = joblib.load("../models/scorecard_v1.joblib")
bp, final = m["binning"], m["final"]

url = URL.create("postgresql+psycopg2", username="postgres",
                 password=getpass.getpass("PostgreSQL şifresi: "),
                 host="localhost", port=5432, database="retail_scorecard")
engine = create_engine(url)

num_cols = [v for v in final if v != "purpose"]
cols = ", ".join(f"{v}::float8 AS {v}" for v in num_cols)
q = f"SELECT id, issue_date, sample, bad, bad_alt, lc_grade, purpose, {cols} FROM clean.lc_model"
df = pd.read_sql(q, engine)

# score every loan with the frozen model
woe = pd.DataFrame({v: bp.get_binned_variable(v).transform(df[v], metric="woe") for v in final}).fillna(0)
df["logit_bad"] = m["intercept"] + woe[final] @ m["coefs"]
df["pd"] = 1 / (1 + np.exp(-df["logit_bad"]))
df["score"] = m["offset"] - m["factor"] * df["logit_bad"]
df["score_1900"] = np.clip(1 + (df["score"] - m["smin"]) / (m["smax"] - m["smin"]) * 1899, 1, 1900).round()

df.groupby("sample")["bad"].agg(["count", "sum", "mean"])

,count,sum,mean
sample,,,
monitoring,0,0.0,NaN
oot,283026,42132.0,0.148863
train,120683,15007.0,0.124351
validation,162570,22315.0,0.137264


In [2]:
from scipy.stats import ks_2samp

grade_num = df["lc_grade"].map({g: i for i, g in enumerate("ABCDEFG", start=1)})

rows = []
for s in ["train", "validation", "oot"]:
    d = df[df["sample"] == s]
    rows.append({
        "sample": s,
        "n": len(d),
        "bad_rate": d["bad"].mean(),
        "gini_model": 2 * roc_auc_score(d["bad"], d["pd"]) - 1,
        "ks_model": ks_2samp(d.loc[d["bad"] == 1, "pd"], d.loc[d["bad"] == 0, "pd"]).statistic,
        "gini_lc_grade": 2 * roc_auc_score(d["bad"], grade_num[d.index]) - 1,
    })
discrimination = pd.DataFrame(rows).round(3)
discrimination

,sample,n,bad_rate,gini_model,ks_model,gini_lc_grade
0,train,120683,0.124,0.328,0.239,0.263
1,validation,162570,0.137,0.323,0.232,0.300
2,oot,283026,0.149,0.336,0.244,0.335


In [3]:
scored = df[df["sample"] != "monitoring"].copy()

overall = (scored.groupby("sample")
           .agg(n=("bad", "size"), predicted_pd=("pd", "mean"), actual_bad=("bad", "mean"))
           .loc[["train", "validation", "oot"]].round(3))
print(overall, "\n")

scored["band"] = pd.cut(scored["score_1900"], [0, 699, 1099, 1499, 1900],
                        labels=["1-699", "700-1099", "1100-1499", "1500-1900"])
(scored.groupby(["sample", "band"], observed=True)
       .agg(n=("bad", "size"), predicted_pd=("pd", "mean"), actual_bad=("bad", "mean"))
       .round(3))

                 n  predicted_pd  actual_bad
sample                                      
train       120683         0.124       0.124
validation  162570         0.131       0.137
oot         283026         0.130       0.149 



n  predicted_pd  actual_bad
sample     band                                       
oot        1-699       39386         0.257       0.281
           700-1099   183221         0.129       0.151
           1100-1499   57830         0.051       0.058
           1500-1900    2589         0.018       0.014
train      1-699       13279         0.253       0.256
           700-1099    80902         0.128       0.128
           1100-1499   25346         0.051       0.050
           1500-1900    1156         0.018       0.010
validation 1-699       22173         0.256       0.260
           700-1099   106857         0.130       0.138
           1100-1499   32196         0.051       0.055
           1500-1900    1344         0.018       0.021